In [1]:
!pip install -q transformers datasets accelerate torch


In [2]:
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset


stories = [
    "Once upon a time, in a small village, there lived a curious girl who loved adventures.",
    "The dark forest whispered secrets as the brave knight walked deeper into the shadows.",
    "In a futuristic city, robots and humans lived together in harmony.",
    "The old man opened the mysterious book and the world around him changed forever."
]


In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Embedding(50257, 768)

In [5]:
stories = [
    "Once upon a time, in a small village, there lived a curious girl who loved adventures.",
    "The dark forest whispered secrets as the brave knight walked deeper into the shadows.",
    "In a futuristic city, robots and humans lived together in harmony.",
    "The old man opened the mysterious book and the world around him changed forever."
]

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

dataset = Dataset.from_dict({"text": stories})
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


In [8]:
training_args = TrainingArguments(
    output_dir="./gpt2-story",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=5e-5,
    fp16=True
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=3.950244585673014, metrics={'train_runtime': 22.5373, 'train_samples_per_second': 0.532, 'train_steps_per_second': 0.266, 'total_flos': 391938048000.0, 'train_loss': 3.950244585673014, 'epoch': 3.0})

In [10]:
def generate_story(prompt, max_length=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "Once upon a time in a magical kingdom,"
print(generate_story(prompt))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time in a magical kingdom, the King is no longer a noble knight, but a young noble knight, a son of a young noble, a son of a noble, an old noble knight, a young noble knight. It is the only time that I was truly a noble knight, and was indeed a king, but my kingdom was only a kingdom. I was a king because I was a noble knight and I was a young noble knight. I could not have been a noble knight because I was a knight, a noble knight. My kingdom was the one who was never a noble
